In [9]:
!pip install -q datasets pandas regex tqdm

import re
import pandas as pd
from datasets import load_dataset, Dataset
from tqdm import tqdm

# Define elderly/geriatric keywords for targeted domain filtering
ELDERLY_KEYWORDS = [
    r'\belderly\b', r'\bgeriatric\b', r'\bolder adult', r'\baging\b',
    r'\bsenior\b', r'\balzheimer', r'\bdementia\b', r'\bparkinson', r'\bosteoporosis\b',
    r'\bfrail\b', r'\bfrailty\b', r'\bpresby\w*', r'\bage-related\b', r'\bmedicare\b',
    r'\bfalls?\b', r'\bcognitive decline\b', r'\bcaregiver\b', r'\bgerontology\b'
]

keyword_regex = re.compile('|'.join(ELDERLY_KEYWORDS), flags=re.IGNORECASE)

def contains_elderly_context(text: str) -> bool:
    """Check if the text contains any elderly/geriatric keywords."""
    if not isinstance(text, str):
        return False
    return bool(keyword_regex.search(text))

def clean_text(text: str) -> str:
    """Clean text by removing HTML tags, extra whitespace, and junk characters."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r'<[^>]+>', ' ', text)  # Clean HTML tags
    text = re.sub(r'\s+', ' ', text)      # Normalize spaces
    text = text.strip()
    return text

processed_datasets = []

### Processing PubMedQA (fedml/PubMedQA_instruction)

In [12]:
print("--> Processing PubMedQA...")
try:
    ds_pubmed = load_dataset("fedml/PubMedQA_instruction", split="train")
    pubmed_list = []

    for row in tqdm(ds_pubmed):
        # Extract fields
        instruction = clean_text(row.get("instruction", "Answer the following medical question using the provided context."))
        input_text = clean_text(row.get("input", ""))
        output_text = clean_text(row.get("output", ""))

        full_text = f"{instruction} {input_text} {output_text}"

        # Filter for elderly care domain
        if contains_elderly_context(full_text):
            pubmed_list.append({
                "source": "PubMedQA",
                "instruction": instruction if instruction else "Answer the medical research query based on context.",
                "input": input_text,
                "response": output_text, # Changed to 'response' as per initial request
                "category": "Medical Q&A", # Added 'category'
            })

    df_pubmed = pd.DataFrame(pubmed_list)
    print(f"PubMedQA Elderly Samples: {len(df_pubmed)}")
    processed_datasets.append(df_pubmed)
except Exception as e:
    print(f"Failed to load/process PubMedQA: {e}")

--> Processing PubMedQA...


README.md:   0%|          | 0.00/1.23k [00:00<?, ?B/s]

data/train-00000-of-00001-d9d142e5f3625d(…): reconstructing file:   0%|          |  0.00B /  274MB            

data/train-00000-of-00001-d9d142e5f3625d(…): downloading bytes:           |  0.00B            

data/test-00000-of-00001-83c33859e732305(…): reconstructing file:   0%|          |  0.00B /  986kB            

data/test-00000-of-00001-83c33859e732305(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/272518 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

100%|██████████| 272518/272518 [00:21<00:00, 12928.08it/s]

PubMedQA Elderly Samples: 7251


### Processing MedQuAD (keivalya/MedQuad-MedicalQnADataset)

In [13]:
print("\n--> Processing MedQuAD...")
try:
    ds_medquad = load_dataset("keivalya/MedQuad-MedicalQnADataset", split="train")
    medquad_list = []

    for row in tqdm(ds_medquad):
        question = clean_text(row.get("Question", ""))
        answer = clean_text(row.get("Answer", ""))
        focus = clean_text(row.get("Focus", ""))

        full_text = f"{focus} {question} {answer}"

        if contains_elderly_context(full_text):
            medquad_list.append({
                "source": "MedQuAD",
                "instruction": f"Provide detailed medical information regarding {focus}." if focus else "Answer the medical query.",
                "input": question,
                "response": answer, # Changed to 'response'
                "category": "Medical Q&A", # Added 'category'
            })

    df_medquad = pd.DataFrame(medquad_list)
    print(f"MedQuAD Elderly Samples: {len(df_medquad)}")
    processed_datasets.append(df_medquad)
except Exception as e:
    print(f"Failed to load/process MedQuAD: {e}")


--> Processing MedQuAD...


README.md:   0%|          | 0.00/233 [00:00<?, ?B/s]

medDataset_processed.csv: reconstructing file:   0%|          |  0.00B / 22.5MB            

medDataset_processed.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/16407 [00:00<?, ? examples/s]

100%|██████████| 16407/16407 [00:07<00:00, 2320.27it/s]

MedQuAD Elderly Samples: 1117


### Processing HealthCareMagic QA (lavita/ChatDoctor-HealthCareMagic-100k)

In [14]:
print("\n--> Processing HealthCareMagic QA...")
try:
    ds_hcm = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")
    hcm_list = []

    for row in tqdm(ds_hcm):
        patient_input = clean_text(row.get("input", ""))
        doctor_output = clean_text(row.get("output", ""))

        full_text = f"{patient_input} {doctor_output}"

        if contains_elderly_context(full_text):
            hcm_list.append({
                "source": "HealthCareMagic",
                "instruction": "Provide a medical consultation and response based on the patient's described symptoms.",
                "input": patient_input,
                "response": doctor_output, # Changed to 'response'
                "category": "Medical Dialogue", # Added 'category'
            })

    df_hcm = pd.DataFrame(hcm_list)
    print(f"HealthCareMagic Elderly Samples: {len(df_hcm)}")
    processed_datasets.append(df_hcm)
except Exception as e:
    print(f"Failed to load/process HealthCareMagic: {e}")


--> Processing HealthCareMagic QA...


README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

data/train-00000-of-00001-5e7cb295b9cff0(…): reconstructing file:   0%|          |  0.00B / 70.5MB            

data/train-00000-of-00001-5e7cb295b9cff0(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]

100%|██████████| 112165/112165 [00:28<00:00, 3964.96it/s]

HealthCareMagic Elderly Samples: 4979


### Processing MedDialog (UCSD26/medical_dialog)

In [15]:
print("\n--> Processing MedDialog...")
try:
    ds_dialog = load_dataset("UCSD26/medical_dialog", split="train")
    dialog_list = []

    for row in tqdm(ds_dialog):
        # MedDialog fields often vary depending on HF format mirror (description/utterance/dialogue)
        context = clean_text(row.get("description", row.get("utterance", "")))
        dialogue = clean_text(row.get("dialogue", row.get("formal_description", "")))

        full_text = f"{context} {dialogue}"

        if contains_elderly_context(full_text):
            dialog_list.append({
                "source": "MedDialog",
                "instruction": "Analyze the patient medical dialogue and provide clinical recommendations.",
                "input": context if context else dialogue[:200],
                "response": dialogue if context else dialogue[200:], # Changed to 'response'
                "category": "Medical Dialogue", # Added 'category'
            })

    df_dialog = pd.DataFrame(dialog_list)
    print(f"MedDialog Elderly Samples: {len(df_dialog)}")
    processed_datasets.append(df_dialog)
except Exception as e:
    print(f"Skipping or handled alternative MedDialog format: {e}")


--> Processing MedDialog...


README.md:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

medical_dialog.py:   0%|          | 0.00/16.3k [00:00<?, ?B/s]

Skipping or handled alternative MedDialog format: Dataset scripts are no longer supported, but found medical_dialog.py


### Unifying, Final Cleaning, and Exporting Datasets

In [16]:
print("\n--> Combining Datasets...")
final_df = pd.concat(processed_datasets, ignore_index=True)

# Drop missing values and duplicates across input/output
final_df.dropna(subset=["input", "response"], inplace=True) # Changed 'output' to 'response'
final_df.drop_duplicates(subset=["input", "response"], inplace=True) # Changed 'output' to 'response'

# Remove trivial/extremely short inputs or outputs
final_df = final_df[(final_df['input'].str.len() > 10) & (final_df['response'].str.len() > 10)] # Changed 'output' to 'response'

print(f"\nFinal Consolidated Elderly Dataset Size: {len(final_df)} records")
print("\nSample Distribution by Source:")
print(final_df['source'].value_counts())

# Export to JSON and Hugging Face Dataset format
# Renaming for clarity as per the initial request of 'medical.json'
final_df.to_json("medical.json", orient="records", indent=2)
hf_dataset = Dataset.from_pandas(final_df)
hf_dataset.save_to_disk("./elderly_dataset_hf")

print("\nSaved output files:")
print("1. medical.json")
print("2. ./elderly_dataset_hf (Hugging Face Dataset folder)")


--> Combining Datasets...

Final Consolidated Elderly Dataset Size: 6083 records

Sample Distribution by Source:
source
HealthCareMagic    4971
MedQuAD            1112
Name: count, dtype: int64


Saving the dataset (0/1 shards):   0%|          | 0/6083 [00:00<?, ? examples/s]


Saved output files:
1. medical.json
2. ./elderly_dataset_hf (Hugging Face Dataset folder)
